# Bagging — bootstrap aggregation for variance reduction

> Tutorial pair for [`bagging.py`](bagging.py).

## 1. Intuition
Take one unstable learner — say a fully grown decision tree, which changes a lot
if you jiggle the data — and train **many copies** of it on different bootstrap
resamples of your training set. Average their predictions (vote for
classification). The individual trees are still wiggly, but their *average* is
smooth: the random errors cancel out. Bias barely changes; **variance drops**.
That is bagging in one sentence.

## 2. Concept (the slide)
- **Bootstrap:** draw $n$ rows *with replacement* → each learner sees a slightly
  different dataset.
- **Aggregate:** mean (regression) or majority vote (classification).
- **Best for high-variance, low-bias learners** (deep trees). For stable
  learners (linear models) it barely helps.
- **OOB error:** the $\approx 37\%$ of rows left out of each bootstrap form a
  built-in validation set.
- **Random forest = bagging + per-split feature subsampling** (the extra step
  that de-correlates the trees).

## 3. Math derivation

**Bootstrap.** A bootstrap sample draws $n$ indices uniformly with replacement.
The chance a particular row is omitted is
$\left(1-\frac1n\right)^n\to e^{-1}\approx0.368$, so each bag uses $\approx63\%$
distinct rows and leaves $\approx37\%$ out-of-bag.

**Variance of an average.** Let $\{T_b\}_{b=1}^B$ be predictors, each with
variance $\sigma^2$ and pairwise correlation $\rho$. Then for the bagged
predictor $\bar T=\frac1B\sum_b T_b$,

$$\operatorname{Var}(\bar T)=\frac1{B^2}\Big(\sum_b\operatorname{Var}(T_b)
  +\sum_{b\ne b'}\operatorname{Cov}(T_b,T_{b'})\Big)
  =\frac{\sigma^2}{B}+\frac{B-1}{B}\rho\sigma^2
  \;\xrightarrow{B\to\infty}\;\rho\,\sigma^2 .$$

- If the predictors were **independent** ($\rho=0$): variance shrinks like
  $\sigma^2/B$ — perfect averaging.
- In practice bootstrap samples overlap, so $\rho>0$ and the variance floors at
  $\rho\sigma^2$. This is *why* random forests add feature subsampling: to push
  $\rho$ down and lower that floor.

**Bias is unchanged.** $\mathbb E[\bar T]=\mathbb E[T]$, so bagging does not fix a
biased learner — it only removes variance. In the bias–variance decomposition
$\mathbb E[(y-\hat f)^2]=\text{bias}^2+\text{variance}+\sigma_\varepsilon^2$,
bagging targets the middle term.

**OOB estimate.** For each sample $i$, average the predictions of only those bags
that did *not* contain $i$; comparing to $y_i$ over all $i$ gives an
(approximately) unbiased test-error estimate without a hold-out split.

## 4. NumPy implementation (generic bagging + OOB + variance experiment)

In [ ]:
# ===== actual implementation from bagging.py =====
from __future__ import annotations

import copy

import numpy as np

SEED = 0

class _Stump:
    """Depth-1 CART (decision stump) — a deliberately weak, biased learner."""

    def __init__(self, task="classification"):
        self.task = task

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        best = (np.inf, None, None, None, None)
        for f in range(X.shape[1]):
            vals = np.unique(X[:, f])
            for t in (vals[:-1] + vals[1:]) / 2 if len(vals) > 1 else []:
                m = X[:, f] <= t
                if m.sum() == 0 or (~m).sum() == 0:
                    continue
                if self.task == "classification":
                    score = self._gini(y[m]) * m.sum() + self._gini(y[~m]) * (~m).sum()
                    lv = self._mode(y[m]); rv = self._mode(y[~m])
                else:
                    score = np.var(y[m]) * m.sum() + np.var(y[~m]) * (~m).sum()
                    lv = y[m].mean(); rv = y[~m].mean()
                if score < best[0]:
                    best = (score, f, t, lv, rv)
        _, self.f, self.t, self.lv, self.rv = best
        if self.f is None:
            self.f, self.t = 0, 0.0
            self.lv = self.rv = (self._mode(y) if self.task == "classification" else y.mean())
        return self

    @staticmethod
    def _gini(y):
        _, c = np.unique(y, return_counts=True); p = c / c.sum()
        return 1 - (p ** 2).sum()

    @staticmethod
    def _mode(y):
        v, c = np.unique(y, return_counts=True); return v[c.argmax()]

    def predict(self, X):
        X = np.asarray(X, float)
        return np.where(X[:, self.f] <= self.t, self.lv, self.rv)

class _Tree:
    """Unconstrained CART (deep) — a low-bias, HIGH-variance learner. Bagging
    shines here. Recursive, self-contained."""

    def __init__(self, task="classification", max_depth=None, min_samples_split=2):
        self.task, self.max_depth, self.min_samples_split = task, max_depth, min_samples_split

    def _imp(self, y):
        if self.task == "classification":
            _, c = np.unique(y, return_counts=True); p = c / c.sum()
            return 1 - (p ** 2).sum()
        return np.var(y) if len(y) else 0.0

    def _leaf(self, y):
        if self.task == "classification":
            v, c = np.unique(y, return_counts=True); return v[c.argmax()]
        return float(y.mean())

    def _best_split(self, X, y):
        # Vectorized prefix-sum split search (O(n log n) per feature).
        n, d = X.shape
        best = (np.inf, None, None)
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs = X[order, f]
            valid = xs[:-1] != xs[1:]
            if not valid.any():
                continue
            cnt_l = np.arange(1, n); cnt_r = n - cnt_l
            if self.task == "classification":
                C = self.n_classes
                oh = np.eye(C)[y[order].astype(int)]
                cum = np.cumsum(oh, axis=0); tot = cum[-1]
                cl = cum[:-1]; cr = tot - cl
                gl = 1 - ((cl / cnt_l[:, None]) ** 2).sum(1)
                gr = 1 - ((cr / cnt_r[:, None]) ** 2).sum(1)
                s = (cnt_l * gl + cnt_r * gr) / n
            else:
                ys = y[order].astype(float)
                cs = np.cumsum(ys)[:-1]; cs2 = np.cumsum(ys ** 2)[:-1]
                tot, tot2 = cs[-1] + ys[-1], cs2[-1] + ys[-1] ** 2
                vl = cs2 / cnt_l - (cs / cnt_l) ** 2
                vr = (tot2 - cs2) / cnt_r - ((tot - cs) / cnt_r) ** 2
                s = (cnt_l * vl + cnt_r * vr) / n
            s = np.where(valid, s, np.inf)
            j = int(np.argmin(s))
            if s[j] < best[0]:
                best = (s[j], f, (xs[j] + xs[j + 1]) / 2)
        return best

    def _build(self, X, y, depth):
        if (len(y) < self.min_samples_split or
                (self.max_depth is not None and depth >= self.max_depth) or
                len(np.unique(y)) == 1):
            return ("leaf", self._leaf(y))
        _, f, t = self._best_split(X, y)
        if f is None:
            return ("leaf", self._leaf(y))
        m = X[:, f] <= t
        return ("node", f, t, self._build(X[m], y[m], depth + 1),
                self._build(X[~m], y[~m], depth + 1))

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        if self.task == "classification":
            self.n_classes = int(y.max()) + 1
        self.root = self._build(X, y, 0)
        return self

    def _one(self, x, node):
        if node[0] == "leaf":
            return node[1]
        _, f, t, l, r = node
        return self._one(x, l if x[f] <= t else r)

    def predict(self, X):
        return np.array([self._one(x, self.root) for x in np.asarray(X, float)])

def variance_experiment(n_trials=20, n_train=150, seed=SEED):
    from sklearn.datasets import make_friedman1
    rng = np.random.default_rng(seed)
    Xte, _ = make_friedman1(n_samples=60, noise=0.0, random_state=999)
    single_preds, bag_preds = [], []
    for t in range(n_trials):
        Xtr, ytr = make_friedman1(n_samples=n_train, noise=1.0,
                                  random_state=int(rng.integers(1 << 30)))
        single = _Tree(task="regression", max_depth=None).fit(Xtr, ytr)
        single_preds.append(single.predict(Xte))
        bag = BaggingNumPy(lambda: _Tree(task="regression"), n_estimators=20,
                           task="regression", seed=t).fit(Xtr, ytr)
        bag_preds.append(bag.predict(Xte))
    # variance of the prediction at each test point, averaged over points
    var_single = np.var(np.array(single_preds), axis=0).mean()
    var_bag = np.var(np.array(bag_preds), axis=0).mean()
    return var_single, var_bag

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_classification, make_friedman1

    # ---------- classification: bag deep trees ----------
    X, y = make_classification(n_samples=500, n_features=10, n_informative=6,
                               n_redundant=2, random_state=SEED)
    Xtr, ytr, Xte, yte = X[:380], y[:380], X[380:], y[380:]
    single = _Tree(task="classification").fit(Xtr, ytr)
    print(f"[clf] single deep tree acc={np.mean(single.predict(Xte) == yte):.3f}")
    bag = BaggingNumPy(lambda: _Tree(task="classification"),
                       n_estimators=50, task="classification").fit(Xtr, ytr)
    print(f"[clf] bagged trees acc={np.mean(bag.predict(Xte) == yte):.3f}  "
          f"OOB acc={bag.oob_score_:.3f}")
    sk = sklearn_reference(Xtr, ytr, n_estimators=50)
    print(f"[clf] sklearn Bagging acc={np.mean(sk.predict(Xte) == yte):.3f}  "
          f"OOB acc={sk.oob_score_:.3f}")

    # ---------- regression ----------
    Xr, yr = make_friedman1(n_samples=400, noise=1.0, random_state=SEED)
    Xrtr, yrtr, Xrte, yrte = Xr[:300], yr[:300], Xr[300:], yr[300:]
    bagr = BaggingNumPy(lambda: _Tree(task="regression"), n_estimators=50,
                        task="regression").fit(Xrtr, yrtr)
    single_r = _Tree(task="regression").fit(Xrtr, yrtr)
    print(f"[reg] single tree MSE={np.mean((single_r.predict(Xrte) - yrte) ** 2):.3f}")
    print(f"[reg] bagged MSE={np.mean((bagr.predict(Xrte) - yrte) ** 2):.3f}  "
          f"OOB MSE={bagr.oob_score_:.3f}")

    # ---------- variance reduction made explicit ----------
    vs, vb = variance_experiment(n_trials=8)
    print(f"[var] mean prediction variance: single tree={vs:.3f}  bagged={vb:.3f}  "
          f"(ratio {vb / vs:.2f}x)")


class BaggingNumPy:
    """Bootstrap-aggregate an arbitrary base learner.

    `base_factory` is a 0-arg callable returning a *fresh* unfitted learner with
    `.fit(X, y)` / `.predict(X)`. We deep-copy it per bag for safety.
    """

    def __init__(self, base_factory, n_estimators=50, task="classification",
                 max_samples=1.0, seed=SEED):
        self.base_factory = base_factory
        self.n_estimators = n_estimators
        self.task = task
        self.max_samples = max_samples
        self.seed = seed
        self.models_ = []
        self.oob_indices_ = []
        self.oob_score_ = None

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        n = len(X)
        m = max(1, int(self.max_samples * n))
        rng = np.random.default_rng(self.seed)
        self.classes_ = np.unique(y) if self.task == "classification" else None
        self.models_, self.oob_indices_ = [], []
        for _ in range(self.n_estimators):
            idx = rng.integers(0, n, m)                       # bootstrap sample
            oob = np.setdiff1d(np.arange(n), np.unique(idx))  # left-out rows
            model = copy.deepcopy(self.base_factory())
            model.fit(X[idx], y[idx])
            self.models_.append(model)
            self.oob_indices_.append(oob)
        self._oob(X, y)
        return self

    def _aggregate(self, preds):
        """preds: (M, n). Vote (clf) or mean (reg) over the M models."""
        if self.task == "classification":
            out = np.empty(preds.shape[1], dtype=self.classes_.dtype)
            for i in range(preds.shape[1]):
                v, c = np.unique(preds[:, i], return_counts=True)
                out[i] = v[c.argmax()]
            return out
        return preds.mean(0)

    def predict(self, X):
        preds = np.array([m.predict(X) for m in self.models_])
        return self._aggregate(preds)

    def _oob(self, X, y):
        n = len(X)
        # collect, for each sample, predictions only from models that didn't see it
        per_sample = [[] for _ in range(n)]
        for model, oob in zip(self.models_, self.oob_indices_):
            if len(oob) == 0:
                continue
            p = model.predict(X[oob])
            for i, idx in enumerate(oob):
                per_sample[idx].append(p[i])
        have = [i for i in range(n) if per_sample[i]]
        if not have:
            self.oob_score_ = None
            return
        if self.task == "classification":
            agg = []
            for i in have:
                v, c = np.unique(per_sample[i], return_counts=True)
                agg.append(v[c.argmax()])
            self.oob_score_ = float(np.mean(np.array(agg) == y[have]))   # OOB acc
        else:
            agg = np.array([np.mean(per_sample[i]) for i in have])
            self.oob_score_ = float(np.mean((agg - y[have]) ** 2))

## 5. Reference / cross-check — why not PyTorch?

Bagging is a *meta-procedure* wrapped around an arbitrary base learner (here,
discrete decision trees / stumps). There is nothing to differentiate at the
ensemble level, so an idiomatic PyTorch model is not the natural tool. We
cross-check against scikit-learn's `Bagging*` estimators.

In [ ]:
# ===== actual implementation from bagging.py =====
def sklearn_reference(X, y, task="classification", **kw):
    from sklearn.ensemble import BaggingClassifier, BaggingRegressor
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    if task == "classification":
        return BaggingClassifier(DecisionTreeClassifier(), oob_score=True,
                                 random_state=SEED, **kw).fit(X, y)
    return BaggingRegressor(DecisionTreeRegressor(), oob_score=True,
                            random_state=SEED, **kw).fit(X, y)

## 6. Train — single tree vs bag, OOB, and an explicit variance experiment

In [ ]:
demo()

## 7. Visualization — averaging smooths a wiggly regressor

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import bagging as M

# 1-D noisy sine: a deep tree wiggles, the bag of trees smooths it out
rng = np.random.default_rng(0)
Xtr = np.sort(rng.uniform(-3, 3, 80)).reshape(-1, 1)
ytr = np.sin(Xtr).ravel() + 0.3 * rng.standard_normal(80)
Xte = np.linspace(-3, 3, 400).reshape(-1, 1)

single = M._Tree(task="regression").fit(Xtr, ytr)
bag = M.BaggingNumPy(lambda: M._Tree(task="regression"), n_estimators=50,
                     task="regression").fit(Xtr, ytr)

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (model, title) in zip(ax, [(single, "single deep tree (high variance)"),
                                   (bag, "bagged 50 trees (variance reduced)")]):
    a.scatter(Xtr, ytr, s=12, c="k", alpha=.5)
    a.plot(Xte, np.sin(Xte), "g--", lw=1, label="true")
    a.plot(Xte, model.predict(Xte), "r", lw=1.5, label="prediction")
    a.set_title(title); a.legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Bagging **reduces variance, not bias** — use it on *overfit-prone* learners
  (deep trees), not on already-stable ones.
- Variance floors at $\rho\sigma^2$; to go lower, **de-correlate** the learners →
  random forest (feature subsampling).
- **OOB error** is a free validation estimate; more bags only help (and cost
  compute), they don't overfit.
- For **bias** reduction, you need sequential error-correction instead →
  boosting (AdaBoost / gradient boosting).